# SP400 + SP500 + SP600 to SPall OOS CNN

Train one I5 model on deduplicated observations before 2020, freeze it, and evaluate on the deduplicated SPall panel from 2020 onward.

## Goal

The train and test panels are independent. Duplicate observations are identified by `sedol + date`; SP400 has first priority, followed by SP500 and SP600.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
from IPython.display import display

import price_trend_spall_oos_cnn as m

print('module:', m.__file__)

### Parameters

In [ ]:
DATA_DIR = Path('.')
TRAIN_FILES = (
    'SP400_picMap5MA_price_chart_image_df.pickle',
    'SP500_picMap5MA_price_chart_image_df.pickle',
    'SP600_picMap5MA_price_chart_image_df.pickle',
)
TEST_FILE = 'SPall_picMap5MA_price_chart_image_df.pickle'
TRAIN_PATTERN = 'SP[456]00_picMap5MA_price_chart_image_df.pickle'
TEST_PATTERN = TEST_FILE

SPLIT_DATE = '2020-01-01'
TEST_END = None             # None uses every available date >= 2020
TRAIN_YEARS = 8             # choose 5, 8, 10, 15, etc.
GATE = 'gate_nan_nan'       # no VIX trading gate by default
SEEDS = 5
EPOCHS = 50
CPU_THREADS = 'auto'

DROP_TRAIN_ABS_RET = None   # examples: 0.10, 0.15, 0.20
TRAIN_VIX_LT = None         # example: 20.0

FILTER_BITS = []
if DROP_TRAIN_ABS_RET is not None:
    FILTER_BITS.append(f'drop_abs{DROP_TRAIN_ABS_RET:g}')
if TRAIN_VIX_LT is not None:
    FILTER_BITS.append(f'train_vix_lt{TRAIN_VIX_LT:g}')
FILTER_TAG = ('_' + '_'.join(FILTER_BITS).replace('.', 'p')) if FILTER_BITS else ''
OUT_DIR = DATA_DIR / f'results_spall_oos_{GATE}{FILTER_TAG}'
RAW_OUT_DIR = DATA_DIR / f'results_spall_oos_{GATE}{FILTER_TAG}_raw'

## Steps

### 1. Validate input files

In [ ]:
manifest = m.spall_input_manifest(
    data_dir=DATA_DIR,
    train_pattern=TRAIN_PATTERN,
    test_pattern=TEST_PATTERN,
)
display(manifest.style.format({'size_gb': '{:.3f}'}))

### 2. Train once and test post-2020

In [ ]:
res = m.run_spall_oos(
    data_dir=DATA_DIR,
    train_pattern=TRAIN_PATTERN,
    test_pattern=TEST_PATTERN,
    split_date=SPLIT_DATE,
    test_end=TEST_END,
    train_years=TRAIN_YEARS,
    gate=GATE,
    horizon=5,
    seeds=SEEDS,
    epochs=EPOCHS,
    cpu_threads=CPU_THREADS,
    drop_train_abs_ret=DROP_TRAIN_ABS_RET,
    train_vix_lt=TRAIN_VIX_LT,
    out_dir=OUT_DIR,
    source_out_dir=RAW_OUT_DIR,
    save_model=True,
    make_plots=True,
    verbose=True,
)

res['out_dir']

## Checks

### 3. Confirm deduplication and time separation

In [ ]:
summary = res['summary']
assert len(summary['train_source_files']) == 3
assert pd.Timestamp(summary['actual_train_date_max']) < pd.Timestamp(SPLIT_DATE)
assert pd.Timestamp(summary['actual_test_date_min']) >= pd.Timestamp(SPLIT_DATE)

audit_fields = [
    'actual_train_date_min', 'actual_train_date_max', 'train_rows_used',
    'train_rows_before_dedup', 'train_rows_after_dedup',
    'train_duplicate_rows_dropped', 'actual_test_date_min',
    'actual_test_date_max', 'test_rows_selected', 'test_rows_before_dedup',
    'test_rows_after_dedup', 'test_duplicate_rows_dropped',
]
display(pd.DataFrame({'value': [summary.get(key) for key in audit_fields]}, index=audit_fields))

### 4. Long-short and long-only results

In [ ]:
performance = pd.DataFrame(
    {
        'Sharpe': [summary['ls_sharpe'], summary['lo_sharpe']],
        'CAGR': [summary['ls_cagr'], summary['lo_cagr']],
    },
    index=['Long-short', 'Long-only'],
)
display(performance.style.format({'Sharpe': '{:.3f}', 'CAGR': '{:.2%}'}))

long_short = pd.read_csv(summary['long_short_csv'])
long_only = pd.read_csv(summary['long_only_csv'])
display(long_short.tail(10))
display(long_only.tail(10))

### 5. VIX diagnostics and plots

In [ ]:
display(res['vix_bucket_summary'])
m.display_results(res)

## Next Steps

Change one training filter or gate at a time, use a new output directory, and compare the untouched post-2020 results.